# Предсказание на своей картинке

Модель выдаёт балл от 0 до 1 и карту, где она видит генерацию. Порог берётся из `runs/pilot/report.md`,
его пишет `scripts/run_test.py`: выше порога значит сгенерирована, при 1 % ложных срабатываний на реальных фото.

In [ ]:
import os, io, re, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.model import Detector

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
size = cfg['train']['size']   # 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)

# порог из отчёта теста; без отчёта 0.5
threshold = 0.5
if (out / 'report.md').exists():
    threshold = float(re.search(r'threshold ([\d.]+)', (out / 'report.md').read_text()).group(1))
threshold

Картинка готовится как на тесте: не-JPEG один раз сжимается в JPEG с качеством 90 (на обучении так сжимались
все не-JPEG), потом картинка целиком сжимается до 512×512. Нормировка ImageNet внутри модели. Модель выдаёт балл
и карту 32×32, по клетке на патч 16×16; карта растягивается обратно на исходную картинку.

In [ ]:
def prepare(path):
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)   # [3, 512, 512], от 0 до 1


@torch.no_grad()
def predict(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        patch_logits, image_logit = model(x)
    prob = torch.sigmoid(image_logit.float()).item()
    heat = torch.sigmoid(patch_logits[0, 0].float()).cpu().numpy()   # карта 32x32
    return prob, heat

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

prob, heat = predict(path)
verdict = 'сгенерирована' if prob > threshold else 'реальная'
print(f'{verdict}: балл {prob:.3f}, порог {threshold:.3f}')

image = Image.open(path).convert('RGB')
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(image)
ax[0].set_title(f'{verdict}: {prob:.3f}, порог {threshold:.3f}')
ax[1].imshow(image)
ax[1].imshow(heat, cmap='jet', alpha=0.45, vmin=0, vmax=1, extent=(0, image.width, image.height, 0), interpolation='bilinear')
ax[1].set_title('где модель видит генерацию')
for a in ax:
    a.axis('off')
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')   # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        prob, heat = predict(p)
        print(f"{prob:.3f}  {'FAKE' if prob > threshold else 'real'}  {p.name}")